# Album Classifier - SageMaker Training & Deployment

This notebook trains and deploys your album classification model.

## Prerequisites

1. ✅ Code and data uploaded to S3 (run `./prepare_for_studio.sh <bucket-name>`)
2. ✅ SageMaker Studio domain created
3. ✅ Update `BUCKET_NAME` below with your bucket name

## Step 1: Install Dependencies

In [ ]:
!pip install sagemaker boto3

## Step 2: Configuration

**IMPORTANT**: Update `BUCKET_NAME` with your actual S3 bucket name!

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch, PyTorchModel
import boto3
import json

# ============================================
# CONFIGURATION - UPDATE THESE VALUES
# ============================================
BUCKET_NAME = 'your-ml-bucket'  # ← CHANGE THIS TO YOUR BUCKET NAME!
REGION = 'us-east-1'  # Change if your bucket is in a different region
ENDPOINT_NAME = 'album-classifier'

# Get execution role (automatically detected in Studio)
role = sagemaker.get_execution_role()
print(f"✓ Using role: {role}")

# Verify bucket access
s3_client = boto3.client('s3', region_name=REGION)
try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"✓ Bucket '{BUCKET_NAME}' is accessible")
    
    # List files
    print("\nS3 Contents:")
    print(f"  Data: s3://{BUCKET_NAME}/data/")
    print(f"  Code: s3://{BUCKET_NAME}/code/")
except Exception as e:
    print(f"✗ Error accessing bucket: {e}")
    print("Make sure the bucket exists and your role has S3 permissions")

## Step 3: Train the Model

This will start a training job. It takes 10-30 minutes depending on instance type.

**Instance Types**:
- `ml.m5.xlarge` - CPU, cheaper (~$0.23/hour)
- `ml.p3.2xlarge` - GPU, faster (~$3.06/hour)

In [ ]:
print("="*50)
print("STEP 1: Training")
print("="*50)

# Create PyTorch estimator
estimator = PyTorch(
    entry_point='train.py',
    source_dir=f's3://{BUCKET_NAME}/code/sourcedir.tar.gz',
    role=role,
    instance_type='ml.m5.xlarge',  # Change to 'ml.p3.2xlarge' for GPU
    instance_count=1,
    framework_version='2.5.0',
    py_version='py311',
    hyperparameters={
        'epochs': '10',
        'batch-size': '16',
        'learning-rate': '0.001',
    },
    output_path=f's3://{BUCKET_NAME}/models/',
    sagemaker_session=sagemaker.Session(),
    base_job_name='album-classifier',
    max_run=86400,  # 24 hours max
)

print("\nStarting training job...")
print(f"  Instance: ml.m5.xlarge")
print(f"  Data: s3://{BUCKET_NAME}/data/")
print(f"  Output: s3://{BUCKET_NAME}/models/")
print("\nThis will take 10-30 minutes...")

# Start training
estimator.fit({'training': f's3://{BUCKET_NAME}/data/'})

print(f"\n✓ Training complete!")
print(f"Model artifacts: {estimator.model_data}")

In [ ]:
print("="*50)
print("STEP 2: Deployment")
print("="*50)

# Create model from training job
model = PyTorchModel(
    model_data=estimator.model_data,
    role=role,
    entry_point='inference.py',
    framework_version='2.5.0',
    py_version='py311',
    source_dir=f's3://{BUCKET_NAME}/code/sourcedir.tar.gz',
)

print("\nDeploying endpoint...")
print("This will take 5-10 minutes...")

# Deploy endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=ENDPOINT_NAME,
    wait=True  # Wait for deployment to complete
)

print(f"\n✓ Endpoint deployed: {ENDPOINT_NAME}")
print(f"Endpoint ARN: {predictor.endpoint}")

## Step 5: Test the Endpoint

Test your deployed endpoint with a sample image.

In [ ]:
print("="*50)
print("STEP 3: Testing")
print("="*50)

# Option 1: Download test image from S3
# Uncomment the line below to download an image
# !aws s3 cp s3://{BUCKET_NAME}/data/images/0.jpg test_image.jpg

try:
    # Read image file
    with open('test_image.jpg', 'rb') as f:
        image_data = f.read()
    
    # Make prediction
    print("\nMaking prediction...")
    response = predictor.predict(
        image_data,
        initial_args={'ContentType': 'image/jpeg'}
    )
    
    print("\nPredictions:")
    print(json.dumps(response, indent=2))
    
except FileNotFoundError:
    print("\n⚠ Test image not found.")
    print("To test:")
    print(f"  1. Download an image: !aws s3 cp s3://{BUCKET_NAME}/data/images/0.jpg test_image.jpg")
    print("  2. Or upload your own test image")
    print("  3. Then run this cell again")
except Exception as e:
    print(f"\n✗ Error testing endpoint: {e}")

## Step 6: Cleanup (Optional)

**Important**: Endpoints cost money while running. Delete when not in use!

In [ ]:
# Uncomment to delete endpoint when done
# predictor.delete_endpoint()
# predictor.delete_model()
# print("✓ Endpoint deleted")

## Next Steps

After deployment:

1. ✅ Test endpoint (Step 5 above)
2. ✅ Deploy API Gateway + Lambda (see `docs/DEPLOY_INFERENCE.md`)
3. ✅ Configure frontend (see `docs/FRONTEND_API_GATEWAY_SETUP.md`)

## Troubleshooting

- **Training fails**: Check CloudWatch logs in SageMaker console
- **Endpoint fails**: Verify model artifacts exist in S3
- **Access denied**: Check IAM role has S3 permissions

See `docs/STUDIO_WALKTHROUGH.md` for detailed troubleshooting.